In [1]:
# 如果本地没有 guided-diffusion，可先：
!git clone https://github.com/openai/guided-diffusion
# 依赖（按需）：
!pip install torch torchvision matplotlib pillow tqdm requests huggingface_hub -U


fatal: destination path 'guided-diffusion' already exists and is not an empty directory.


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os, sys, math, random
from types import SimpleNamespace
from typing import Tuple, Optional

# >>> 若 guided-diffusion 不在 PYTHONPATH，请取消注释并改为你的路径
sys.path.append("./guided-diffusion")

In [9]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.2 MB/s eta 0:00:00


In [16]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.transforms import functional as TF

from accelerate import Accelerator
from diffusers import DDPMScheduler
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from accelerate.utils import GradScalerKwargs

# ------- guided-diffusion 引用 -------
try:
    from guided_diffusion import dist_util
    from guided_diffusion.script_util import model_and_diffusion_defaults, create_model_and_diffusion
except Exception as e:
    raise RuntimeError(
        "未能导入 guided-diffusion，请先 git clone 并将路径加入 PYTHONPATH。\n"
        "仓库：https://github.com/openai/guided-diffusion"
    ) from e


In [19]:
@torch.no_grad()
def save_image_grid(tensor_bchw: torch.Tensor, path: str, nrow: int = 4):
    """
    tensor_bchw: [B,3,H,W]，数值范围[-1,1]
    """
    import numpy as np
    from PIL import Image
    t = tensor_bchw.clamp(-1, 1)
    t = (t * 0.5 + 0.5).detach().cpu().numpy()  # [B, C, H, W] in [0,1]
    B, C, H, W = t.shape
    rows = (B + nrow - 1) // nrow
    canvas = np.ones((rows * H, nrow * W, C), dtype=np.float32)
    for i in range(B):
        r, c = i // nrow, i % nrow
        canvas[r*H:(r+1)*H, c*W:(c+1)*W] = np.transpose(t[i], (1, 2, 0))
    Image.fromarray((canvas * 255).astype(np.uint8)).save(path)

In [20]:
# =========================
# 采样（像素域 DDPM，步步重置上下文）
# =========================
@torch.no_grad()
def sample_inpaint_ddpm(unet: GuidedUNetWrapper,
                        scheduler: DDPMScheduler,
                        img: torch.Tensor,      # [B,3,H,W] in [-1,1]
                        mask2d: torch.Tensor,   # [B,1,H,W] ∈ {0,1}
                        steps: int = 250) -> torch.Tensor:
    device = img.device
    B = img.shape[0]
    scheduler.set_timesteps(steps, device=device)

    # 初始化：mask 内纯噪，mask 外 = 原图
    x = torch.randn_like(img) * mask2d + img * (1.0 - mask2d)

    for t in scheduler.timesteps:
        eps = unet(x, t.expand(B), encoder_hidden_states=None).sample
        step = scheduler.step(eps, t, x)
        x = step.prev_sample
        # 重新覆盖上下文
        x = x * mask2d + img * (1.0 - mask2d)
    return x.clamp(-1, 1)

In [17]:
# -*- coding: utf-8 -*-
# 训练：像素空间 512×512（无 VAE），预训练 UNet = OpenAI guided-diffusion 512x512
# - 仅在 bbox 内加噪并监督噪声；mask 外保持干净
# - 兼容你原有的 CondEncoder + bbox 坐标接口（但 guided-diffusion UNet 不支持 cross-attn，条件被忽略）
# 依赖：
#   pip install torch torchvision accelerate diffusers tqdm pillow matplotlib requests huggingface_hub
#   git clone https://github.com/openai/guided-diffusion
# =========================
# 工具：下载预训练 512×512 权重
# =========================
def download_openai_512(dst_dir="./models") -> str:
    import requests
    os.makedirs(dst_dir, exist_ok=True)
    url = "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/512x512_diffusion.pt"
    dst = os.path.join(dst_dir, "512x512_diffusion.pt")
    if os.path.isfile(dst):
        print(f"[download] 已存在: {dst}")
        return dst
    print(f"[download] 正在下载 OpenAI 512x512 -> {dst}")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        cur = 0
        with open(dst, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    cur += len(chunk)
                    if total:
                        print(f"\r  {cur/total*100:5.1f}%", end="")
    print("\n[download] 完成")
    return dst

def download_hf_uncond_512(dst_dir="./models") -> str:
    # 无条件 512×512（社区从 OpenAI 512×512 微调）
    from huggingface_hub import snapshot_download
    os.makedirs(dst_dir, exist_ok=True)
    local_dir = snapshot_download(
        repo_id="lowlevelware/512x512_diffusion_unconditional_ImageNet",
        local_dir=os.path.join(dst_dir, "hf_uncond_512"),
        local_dir_use_symlinks=False
    )
    # 取第一个 .pt
    for f in os.listdir(local_dir):
        if f.endswith(".pt"):
            ckpt = os.path.join(local_dir, f)
            print("[download] HF 无条件权重:", ckpt)
            return ckpt
    raise FileNotFoundError("HF 仓库中未找到 .pt 文件")


# =========================
# 你已有的模块（保持接口）
# =========================
class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1
    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)
    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1,N,D]
    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()
    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    """
    像素域 masked image -> 64 个 token（每个 736 维）
    （与原代码保持一致；guided-diffusion UNet 不使用这些 token，但保留接口以最小侵入替换）
    """
    def __init__(self, in_channels=3, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64),   # [B,64,512,512]
            nn.AvgPool2d(4),                  # [B,64,128,128]
            ResidualBlock(64, 128),
            nn.AvgPool2d(4),                  # [B,128,32,32]
            ResidualBlock(128, 256),
            nn.AvgPool2d(4),                  # [B,256,8,8]
            nn.Conv2d(256, out_channels, kernel_size=1)  # [B,736,8,8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),        # [B,736,64]
            Transpose(-1, -2),    # [B,64,736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)
    def forward(self, x):
        feat = self.encoder(x)         # [B,736,8,8]
        tokens = self.proj(feat)       # [B,64,736]
        tokens = self.pos_embed(tokens)
        tokens = self.norm(tokens)
        return tokens  # [B,64,736]


# =========================
# 数据集（修正 mask 写法）
# =========================
class OutpaintDataset(Dataset):
    def __init__(self, root_dir, img_size=512, masks_per_image=100):
        self.img_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                          if f.lower().endswith((".png",".jpg",".jpeg",".bmp",".webp"))]
        self.img_files.sort()
        self.img_size = img_size
        self.masks_per_image = masks_per_image
        self.transform = transforms.Compose([
            transforms.RandomChoice([
                transforms.Lambda(lambda x: x),
                transforms.Lambda(lambda x: TF.rotate(x, 90)),
                transforms.Lambda(lambda x: TF.rotate(x, 180)),
                transforms.Lambda(lambda x: TF.rotate(x, 270))
            ]),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)  # -> [-1,1]
        ])

    def __len__(self):
        return max(1, len(self.img_files)) * self.masks_per_image

    def _gen_random_bbox(self):
        if random.random() < 0.5:
            aspect_ratio = random.uniform(1.0, 33.0)
        else:
            aspect_ratio = random.uniform(0.03, 1.0)
        area = random.uniform(0.1, 0.5) ** 2
        w = min(np.sqrt(area * aspect_ratio), 0.99)
        h = min(np.sqrt(area / aspect_ratio), 0.99)
        x1 = random.uniform(0.05, 0.99 - w)
        y1 = random.uniform(0.05, 0.99 - h)
        return (torch.tensor(x1, dtype=torch.float32),
                torch.tensor(y1, dtype=torch.float32),
                torch.tensor(x1 + w, dtype=torch.float32),
                torch.tensor(y1 + h, dtype=torch.float32))

    def __getitem__(self, idx):
        img_idx = idx // self.masks_per_image
        img = Image.open(self.img_files[img_idx % len(self.img_files)]).convert("RGB")
        img = self.transform(img)  # [3,H,W] in [-1,1]

        bbox = self._gen_random_bbox()
        _, H, W = img.shape
        x1 = int(bbox[0].item() * W); y1 = int(bbox[1].item() * H)
        x2 = int(bbox[2].item() * W); y2 = int(bbox[3].item() * H)
        mask = torch.zeros((1, H, W), dtype=torch.float32)
        mask[:, y1:y2, x1:x2] = 1.0  # 注意顺序

        masked_img = img * (1.0 - mask)
        return masked_img, img, torch.stack(bbox), mask


# =========================
# 包装：guided-diffusion UNet 适配原调用
# =========================
class GuidedUNetWrapper(nn.Module):
    """
    适配调用签名：forward(x, t, encoder_hidden_states=None) -> .sample
    实际忽略 encoder_hidden_states（guided-diffusion 不支持 cross-attn）
    """
    def __init__(self, gd_unet, learn_sigma=True):
        super().__init__()
        self.gd_unet = gd_unet
        self.learn_sigma = learn_sigma
    def forward(self, x, timesteps, encoder_hidden_states=None):
        # guided-diffusion 前向：返回 [C 或 2C, H, W]
        eps = self.gd_unet(x, timesteps)
        if self.learn_sigma and eps.shape[1] == x.shape[1] * 2:
            eps = eps[:, :x.shape[1]]  # 只取 epsilon
        return SimpleNamespace(sample=eps)


# =========================
# 训练器（像素域 + 预训练 UNet）
# =========================
class OutpaintTrainer_GD512:
    def __init__(self,
                 train_root="drive/MyDrive/resize1",
                 val_root="drive/MyDrive/resizeval1",
                 use_openai=True,    # True: OpenAI 512×512；False: HF 无条件 512×512
                 ckpt_path: Optional[str]=None,
                 img_size=512,
                 batch_size=4,
                 lr=2e-5,
                 save_dir="drive/MyDrive/checkpoint_pixel512"):
        scaler_kwargs = GradScalerKwargs(enabled=False)
        self.accelerator = Accelerator(mixed_precision='fp16', kwargs_handlers=[scaler_kwargs])
        self.loss_history, self.val_loss_history = [], []
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)

        # Data
        train_data = OutpaintDataset(train_root, img_size=img_size)
        val_data   = OutpaintDataset(val_root,   img_size=img_size)
        self.train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
        self.val_loader   = DataLoader(val_data,   batch_size=max(1,batch_size//2), shuffle=False, num_workers=2, pin_memory=True)

        # guided-diffusion 模型 & diffusion
        defaults = model_and_diffusion_defaults()
        defaults.update(dict(
            image_size=img_size,
            num_channels=256,
            num_res_blocks=2,
            learn_sigma=True,
            class_cond=False,             # 使用无条件；若要 class-conditional，可改 True 并提供 y
            use_scale_shift_norm=True,
            attention_resolutions="32,16,8",
            dropout=0.0,
            resblock_updown=True,
            use_fp16=True,
        ))
        gd_model, gd_diffusion = create_model_and_diffusion(**defaults)
        gd_model.convert_to_fp16()

        # 下载/加载预训练权重
        if ckpt_path is None or not os.path.isfile(ckpt_path):
            ckpt_path = download_openai_512() if use_openai else download_hf_uncond_512()
        state = torch.load(ckpt_path, map_location="cpu")
        missing, unexpected = gd_model.load_state_dict(state, strict=False)
        print("[load] missing:", missing)
        print("[load] unexpected:", unexpected)

        # 包装成与原 unet 相同的 forward 接口
        self.unet = GuidedUNetWrapper(gd_model, learn_sigma=True)

        # 条件模块：保留（但 UNet 不消费）
        self.cond_proj = CondEncoder(in_channels=3, out_channels=736, num_tokens=64)
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # 噪声调度（像素域）
        self.noise_scheduler = DDPMScheduler(
            num_train_timesteps=gd_diffusion.num_timesteps,  # 1000
            beta_schedule="squaredcos_cap_v2",
            prediction_type="epsilon"
        )

        # 优化器
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.cond_proj.parameters()) +
            list(self.coord_encoder.parameters()),
            lr=lr
        )

        # accelerate 封装
        comps = [self.unet, self.cond_proj, self.coord_encoder, self.optimizer, self.train_loader, self.val_loader]
        self.unet, self.cond_proj, self.coord_encoder, self.optimizer, self.train_loader, self.val_loader = \
            self.accelerator.prepare(*comps)

        self.accelerator.register_for_checkpointing(self.unet, self.cond_proj, self.coord_encoder, self.optimizer)

        def visualize_val(self, epoch: int, max_items: int = 4, steps: int = 150):
        """
        从 val_loader 抽一批做 inpainting，可视化：
        行顺序：masked_input（用-1表示掩码区） | 原图 | 生成结果
        """
        self.unet.eval()
        try:
            batch = next(iter(self.val_loader))
        except StopIteration:
            self.unet.train()
            return

        masked_imgs, target_imgs, bbox, mask_2d = batch
        device = self.accelerator.device
        masked_imgs = masked_imgs[:max_items].to(device)
        target_imgs = target_imgs[:max_items].to(device)
        mask_2d     = mask_2d[:max_items].to(device)

        # 采样生成
        with torch.no_grad():
            out = sample_inpaint_ddpm(
                unet=self.unet,
                scheduler=self.noise_scheduler,
                img=target_imgs,
                mask2d=mask_2d,
                steps=steps
            )
                # 可视化：把被遮挡的区域画成 -1（黑），方便看上下文
        masked_vis = target_imgs * (1.0 - mask_2d) + (-1.0) * mask_2d
        # 拼接：masked_vis | gt | out
        grid = torch.cat([masked_vis, target_imgs, out], dim=0)

        if self.accelerator.is_main_process:
            os.makedirs(self.save_dir, exist_ok=True)
            save_path = os.path.join(self.save_dir, f"val_epoch_{epoch:04d}.png")
            save_image_grid(grid, save_path, nrow=max_items)
            self.accelerator.print(f"[viz] saved: {save_path}")

        self.unet.train()

    def validate_step(self):
        self.unet.eval()
        tot, n = 0.0, 0
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validating"):
                loss = self.train_step(batch)
                tot += loss.item(); n += 1
        self.unet.train()
        return tot / max(1,n)

    def train_step(self, batch):
        masked_imgs, target_imgs, bbox, mask_2d = batch  # mask_2d:[B,1,H,W]
        bbox = bbox.float()
        B, C, H, W = target_imgs.shape
        device = target_imgs.device

        # 采样噪声 & 时间步
        noise = torch.randn_like(target_imgs)
        timesteps = torch.randint(0, self.noise_scheduler.config.num_train_timesteps, (B,), device=device, dtype=torch.long)

        # 在 mask 内加噪；mask 外保持干净
        noisy_full = self.noise_scheduler.add_noise(target_imgs, noise, timesteps)
        noisy_imgs = target_imgs * (1.0 - mask_2d) + noisy_full * mask_2d

        #（保留接口）条件：masked image -> tokens；bbox -> 32；拼 768 维，但 UNet 实际忽略
        coord_feat = self.coord_encoder(bbox)                                   # [B,32]
        _tokens = self.cond_proj(masked_imgs)                                   # [B,64,736]
        _cond = torch.cat([_tokens, coord_feat.unsqueeze(1).expand(-1, 64, -1)], dim=-1)  # [B,64,768]

        # 预测噪声
        noise_pred = self.unet(noisy_imgs, timesteps, encoder_hidden_states=_cond).sample  # [B,3,H,W]

        # 仅在掩码内计算 MSE
        loss = F.mse_loss(noise_pred * mask_2d, noise * mask_2d)
        return loss

    def train(self, epochs=300, patience=5):
        best_val, patience_counter = float('inf'), 0
        for epoch in range(epochs):
            for param_group in self.optimizer.param_groups:
                self.accelerator.print(f"Epoch {epoch}: lr = {param_group['lr']}")
            self.unet.train()
            progress_bar = tqdm(self.train_loader, desc=f"Epoch {epoch + 1} [Training]")
            epoch_losses = []

            for batch in progress_bar:
                with self.accelerator.accumulate(self.unet):
                    loss = self.train_step(batch)
                    self.accelerator.backward(loss)
                    if self.accelerator.sync_gradients:
                        self.accelerator.clip_grad_norm_(self.unet.parameters(), 1.0)
                    self.optimizer.step()
                    self.optimizer.zero_grad()
                epoch_losses.append(loss.item())

            epoch_avg = float(np.mean(epoch_losses)) if epoch_losses else 0.0
            self.loss_history.append(epoch_avg)
            self.visualize_val(epoch)
            val_loss = self.validate_step()
            self.val_loss_history.append(val_loss)
            self.accelerator.print(f"Epoch {epoch+1}: train={epoch_avg:.6f}  val={val_loss:.6f}")

            if val_loss < best_val:
                best_val = val_loss; patience_counter = 0
                self.accelerator.wait_for_everyone()
                self.accelerator.save_state(self.save_dir)
            else:
                patience_counter += 1
                self.accelerator.print(f"Early Stopping Counter: {patience_counter}/{patience}")

            if patience_counter >= patience:
                self.accelerator.print("Early stopping triggered.")
                self.accelerator.wait_for_everyone()
                self.accelerator.save_state(self.save_dir)
                break

            if epoch % 2 == 0 and epoch > 0:
                for g in self.optimizer.param_groups:
                    g['lr'] *= 0.5
                    self.accelerator.print(f"Decay LR -> {g['lr']}")

            self._plot_loss_curve()

    def _plot_loss_curve(self):
        plt.figure(figsize=(8,6))
        plt.plot(range(1, len(self.loss_history)+1), self.loss_history, label="Training Loss", marker='o')
        plt.plot(range(1, len(self.val_loss_history)+1), self.val_loss_history, label="Validation Loss", marker='s')
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training and Validation Loss (Pixel Space, GD 512×512)")
        plt.legend(); plt.grid(True); plt.show(); plt.close()






In [18]:
# 运行
# =========================
if __name__ == "__main__":
    # 使用前，确保：
    # 1) guided-diffusion 已克隆并在 PYTHONPATH
    # 2) 数据集 train_root / val_root 里是 512×512 RGB 图（或脚本会自动 resize）
    trainer = OutpaintTrainer_GD512(
        train_root="drive/MyDrive/resize1",
        val_root="drive/MyDrive/resizeval1",
        use_openai=True,          # True: OpenAI 官方 512×512；False: HF 无条件 512×512
        ckpt_path=None,           # 也可显式给出 ckpt 路径；为 None 时自动下载
        img_size=512,
        batch_size=2,             # 像素域更吃显存，可按显卡调整
        lr=2e-5,
        save_dir="drive/MyDrive/checkpoint_pixel512"
    )
    trainer.train(epochs=300, patience=5)

    # # 推理示例（可选）：
    # # 假设你有一批验证样本
    # batch = next(iter(trainer.val_loader))
    # masked_imgs, target_imgs, bbox, mask2d = [x.to(trainer.accelerator.device) for x in batch]
    # out = sample_inpaint_ddpm(trainer.unet, trainer.noise_scheduler, target_imgs, mask2d, steps=250)
    # # out 即修补后的结果（范围 [-1,1]）


[download] 已存在: ./models/512x512_diffusion.pt
[load] missing: []
[load] unexpected: ['label_emb.weight']
Epoch 0: lr = 2e-05


Epoch 1 [Training]:   0%|          | 0/3300 [00:00<?, ?it/s]

KeyboardInterrupt: 